Run from the repository root or `notebooks/`; no other notebook or live kernel state is required. 

Inputs are the `stats_target.pkl` and `in_indices_target.pkl` files beneath `pp_data/` for the configured settings and seeds. 

Requires NumPy, Matplotlib, and tueplots.

For each budget M, evaluate M+1 models leave-one-out, fitting per-record IN/OUT Gaussians with the remaining M models.

Configure inputs and output below. Full evaluation recomputes the scores and can require substantial memory. 

In [ ]:
from pathlib import Path
import pickle
import numpy as np
from tueplots import bundles
import matplotlib.pyplot as plt
from cycler import cycler

rc = bundles.iclr2024(usetex=False)
# Match the source figure colors.
palette = [
    "#E69F00",  # Orange
    "#56B4E9",  # Sky blue
    "#000000",  # Black
    "#009E73",  # Bluish green
    "#F0E442",  # Yellow
    "#0072B2",  # Blue
    "#D55E00",  # Vermilion
    "#CC79A7",  # Purple

]
rc.update({
    "axes.prop_cycle": cycler(color=palette),
    "legend.frameon": False,
    "axes.grid": False,
})

In [ ]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SEEDS = [42, 100, 125]
ALPHA = 0.01
FPC = 0.5
Ms = [2**i for i in range(4, 13)]
N_MODELS = max(Ms) + 1
PANEL_SETTINGS = {
    "TabPFN": ("adult_balanced", "adult", 10000),
    "CIFAR10 + FiLM": ("cifar10/film", "cifar10", 1000),
    "CIFAR10 + Head": ("cifar10/head", "cifar10", 1000),
}
OUTPUT_PATH = ROOT / "plots" / f"lira_pp_fpc_fpr_{ALPHA}.pdf"
SAVE_FIGURE = True

In [ ]:
def get_llr(t, mu_in, mu_out, sigma_in, sigma_out, eps = 1e-12):
    sigma_in  = np.maximum(sigma_in,  eps)
    sigma_out = np.maximum(sigma_out, eps)
    return np.log(sigma_out) - np.log(sigma_in) + 0.5 * (t - mu_out)**2/ sigma_out**2 - 0.5 * (t - mu_in)**2/ sigma_in**2

def average_tpr_at_max_m(LLRs_original, LLRs, all_in_indices):
    # Average per-sample TPR from original (uncorrected) LiRA at maximum M.
    # Each record gets its own empirical OUT threshold at FPR = ALPHA.
    M_MAX = max(Ms)
    llrs_max = LLRs_original[M_MAX]
    in_max = all_in_indices[:M_MAX + 1, :]
    valid_max = np.isfinite(llrs_max)
    out_max = ~in_max & valid_max
    in_valid_max = in_max & valid_max

    tau_per_sample_max = np.nanquantile(
        np.where(out_max, llrs_max, np.nan), 1 - ALPHA, axis=0)
    tpr_per_sample_max = safe_divide(
        ((llrs_max > tau_per_sample_max) & in_valid_max).sum(axis=0),
        in_valid_max.sum(axis=0))
    tpr_per_sample_max = np.where(
        np.isfinite(tau_per_sample_max), tpr_per_sample_max, np.nan)
    AVG_TPR_SAMPLE_MAX = np.nanmean(tpr_per_sample_max)

    # FPC-adjusted average per-sample TPR, using LLRs directly (no PP).
    llrs_max_fpc = LLRs[M_MAX]
    valid_max_fpc = np.isfinite(llrs_max_fpc)
    out_max_fpc = ~in_max & valid_max_fpc
    in_valid_max_fpc = in_max & valid_max_fpc

    tau_per_sample_max_fpc = np.nanquantile(
        np.where(out_max_fpc, llrs_max_fpc, np.nan), 1 - ALPHA, axis=0)
    tpr_per_sample_max_fpc = safe_divide(
        ((llrs_max_fpc > tau_per_sample_max_fpc) & in_valid_max_fpc).sum(axis=0),
        in_valid_max_fpc.sum(axis=0))
    tpr_per_sample_max_fpc = np.where(
        np.isfinite(tau_per_sample_max_fpc), tpr_per_sample_max_fpc, np.nan)
    AVG_TPR_SAMPLE_MAX_FPC = np.nanmean(tpr_per_sample_max_fpc)
    return AVG_TPR_SAMPLE_MAX, AVG_TPR_SAMPLE_MAX_FPC


EPS = 1e-12

def safe_divide(num, den, fill=np.nan):
    """
    Elementwise num / den, but returns `fill` wherever den == 0.
    """
    num = np.asarray(num, dtype=float)
    den = np.asarray(den)
    return np.divide(
        num,
        den,
        out=np.full_like(num, fill, dtype=float),
        where=(den != 0)
    )


def evaluate_seed(seed, data_dir, dataset, n_records):
    budgets = Ms
    if dataset == "cifar10":
        DATA_PATH = ROOT / "pp_data" / data_dir / f"Seed={seed}" / "T=100"
    elif dataset == "adult":
        DATA_PATH = ROOT / "pp_data" / data_dir / f"Seed={seed}" / "T=10000"
    with open(DATA_PATH / "in_indices_target.pkl", "rb") as f:
        in_indices = pickle.load(f)[:N_MODELS, :n_records]
    with open(DATA_PATH / "stats_target.pkl", "rb") as f:
        stats = pickle.load(f)

    if len(stats.shape) == 3:
        stats = stats[:N_MODELS, :n_records, 0]
    else:
        stats = stats[:N_MODELS, :n_records]

    assert in_indices.shape == stats.shape == (N_MODELS, n_records)
    all_stats = np.copy(stats).astype(float, copy=False)
    all_in_indices = np.asarray(in_indices, dtype=bool).copy()

    LLRs = {}
    STATS_PP = {}
    LLRs_original = {}
    STATS_PP_original = {}

    for m in range(len(budgets)):
        M_total = budgets[m]

        stats = all_stats[:M_total + 1, :]
        in_indices = all_in_indices[:M_total + 1, :]

        M_current, N_current = stats.shape
        assert in_indices.shape == (M_current, N_current), "in_indices must be (M, N)"

        in_mask = in_indices.astype(bool)
        out_mask = ~in_mask

        # ------------------------------------------------------------------
        # Element-wise masks
        # ------------------------------------------------------------------
        stats_sq = stats ** 2

        stats_in = np.where(in_mask, stats, 0.0)
        stats_out = np.where(out_mask, stats, 0.0)

        stats_in_sq = np.where(in_mask, stats_sq, 0.0)
        stats_out_sq = np.where(out_mask, stats_sq, 0.0)

        col_sum_in = stats_in.sum(axis=0)
        col_sum_out = stats_out.sum(axis=0)

        col_sum_in_sq = stats_in_sq.sum(axis=0)
        col_sum_out_sq = stats_out_sq.sum(axis=0)

        col_cnt_in = in_mask.sum(axis=0)
        col_cnt_out = out_mask.sum(axis=0)

        # ------------------------------------------------------------------
        # LOO sums / counts
        # ------------------------------------------------------------------
        loo_cnt_in = col_cnt_in - in_mask.astype(int)
        loo_cnt_out = col_cnt_out - out_mask.astype(int)

        loo_sum_in = col_sum_in - stats_in
        loo_sum_out = col_sum_out - stats_out

        loo_sum_in_sq = col_sum_in_sq - stats_in_sq
        loo_sum_out_sq = col_sum_out_sq - stats_out_sq

        # ------------------------------------------------------------------
        # Divide-by-zero-safe LOO means
        # Empty LOO groups get nan.
        # ------------------------------------------------------------------
        mu_ins = safe_divide(loo_sum_in, loo_cnt_in)
        mu_outs = safe_divide(loo_sum_out, loo_cnt_out)

        # ------------------------------------------------------------------
        # Divide-by-zero-safe second moments and variances
        # ------------------------------------------------------------------
        ex2_ins = safe_divide(loo_sum_in_sq, loo_cnt_in)
        ex2_outs = safe_divide(loo_sum_out_sq, loo_cnt_out)

        var_ins = ex2_ins - mu_ins ** 2
        var_outs = ex2_outs - mu_outs ** 2

        # Numerical cleanup:
        # - tiny negative values from floating-point cancellation go to 0
        # - truly invalid entries stay nan
        var_ins = np.where(loo_cnt_in > 0, np.maximum(var_ins, 0.0), np.nan)
        var_outs = np.where(loo_cnt_out > 0, np.maximum(var_outs, 0.0), np.nan)

        # Correct both reference variances and the held-out evaluation statistics.
        var_ins_fpc = var_ins / FPC
        var_outs_fpc = var_outs / FPC
        # Evaluation-only correction: true membership selects the LOO class mean.
        # Rescale residuals deterministically before applying corrected scores.
        mu_class = np.where(in_mask, mu_ins, mu_outs)
        stats_fpc = mu_class + (stats - mu_class) / np.sqrt(FPC)
        sigma_ins = np.sqrt(var_ins_fpc)
        sigma_outs = np.sqrt(var_outs_fpc)

        # ------------------------------------------------------------------
        # Safe sigmas for likelihood calculations
        # If count is valid but variance is zero, use EPS instead of 0.
        # If count is invalid, keep nan.
        # ------------------------------------------------------------------
        sigma_ins_safe = np.where(
            loo_cnt_in > 0,
            np.maximum(sigma_ins, EPS),
            np.nan
        )

        sigma_outs_safe = np.where(
            loo_cnt_out > 0,
            np.maximum(sigma_outs, EPS),
            np.nan
        )

        valid_in = (
            (loo_cnt_in > 0)
            & np.isfinite(mu_ins)
            & np.isfinite(sigma_ins_safe)
        )

        valid_out = (
            (loo_cnt_out > 0)
            & np.isfinite(mu_outs)
            & np.isfinite(sigma_outs_safe)
        )

        valid_llr = valid_in & valid_out

        # ------------------------------------------------------------------
        # Computing LLR grid
        # ------------------------------------------------------------------
        with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
            llr_grid = get_llr(
                stats_fpc,
                mu_ins,
                mu_outs,
                sigma_ins_safe,
                sigma_outs_safe
            )

        # Mark entries with insufficient LOO data as nan.
        llr_grid = np.where(valid_llr, llr_grid, np.nan)

        LLRs[M_total] = llr_grid

        # ------------------------------------------------------------------
        # Computing PP Stats Grid
        # ------------------------------------------------------------------
        delta_mu = mu_ins - mu_outs

        direction = np.where(delta_mu >= 0, 1.0, -1.0)
        numerator = direction * (stats_fpc - mu_outs)

        valid_pp = (
            valid_out
            & np.isfinite(delta_mu)
            & np.isfinite(numerator)
        )

        stats_pp = np.divide(
            numerator,
            sigma_outs_safe,
            out=np.full_like(stats, np.nan, dtype=float),
            where=valid_pp
        )

        STATS_PP[M_total] = stats_pp

        # Original baselines: unperturbed observations and uncorrected LOO variances.
        sigma_ins_original = np.maximum(np.sqrt(var_ins), EPS)
        sigma_outs_original = np.maximum(np.sqrt(var_outs), EPS)
        with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
            llrs_original = get_llr(stats, mu_ins, mu_outs,
                                    sigma_ins_original, sigma_outs_original)
            pp_original = direction * (stats - mu_outs) / sigma_outs_original
        LLRs_original[M_total] = np.where(valid_llr, llrs_original, np.nan)
        STATS_PP_original[M_total] = np.where(
            valid_out & np.isfinite(delta_mu), pp_original, np.nan)

    TAUS_ALL = np.zeros(len(budgets))
    TAUS_ALL_PP = np.zeros(len(budgets))
    TAUS_ALL_original = np.zeros(len(budgets))
    TAUS_ALL_PP_original = np.zeros(len(budgets))
    for i, M_i in enumerate(budgets):
        mask_out = ~all_in_indices[:M_i + 1, :]
        llrs = LLRs[M_i]
        pp = STATS_PP[M_i]
        TAUS_ALL[i] = np.quantile(llrs[mask_out & np.isfinite(llrs)], 1 - ALPHA)
        TAUS_ALL_PP[i] = np.quantile(pp[mask_out & np.isfinite(pp)], 1 - ALPHA)
        llrs_original = LLRs_original[M_i]
        pp_original = STATS_PP_original[M_i]
        TAUS_ALL_original[i] = np.quantile(
            llrs_original[mask_out & np.isfinite(llrs_original)], 1 - ALPHA)
        TAUS_ALL_PP_original[i] = np.quantile(
            pp_original[mask_out & np.isfinite(pp_original)], 1 - ALPHA)

    TPR_concat = np.zeros(len(budgets))
    TPR_concat_pp = np.zeros(len(budgets))
    TPR_concat_original = np.zeros(len(budgets))
    TPR_concat_pp_original = np.zeros(len(budgets))
    for i, M_i in enumerate(budgets):
        mask_in = all_in_indices[:M_i + 1, :]
        llrs = LLRs[M_i]
        pp = STATS_PP[M_i]
        valid_llr = np.isfinite(llrs)
        valid_pp = np.isfinite(pp)
        pred_llr = llrs > TAUS_ALL[i]
        pred_pp = pp > TAUS_ALL_PP[i]

        TPR_concat[i] = (pred_llr & mask_in & valid_llr).sum() / (mask_in & valid_llr).sum()
        TPR_concat_pp[i] = (pred_pp & mask_in & valid_pp).sum() / (mask_in & valid_pp).sum()

        llrs_original = LLRs_original[M_i]
        pp_original = STATS_PP_original[M_i]
        valid_llr_original = mask_in & np.isfinite(llrs_original)
        valid_pp_original = mask_in & np.isfinite(pp_original)
        TPR_concat_original[i] = (
            (llrs_original > TAUS_ALL_original[i]) & valid_llr_original
        ).sum() / valid_llr_original.sum()
        TPR_concat_pp_original[i] = (
            (pp_original > TAUS_ALL_PP_original[i]) & valid_pp_original
        ).sum() / valid_pp_original.sum()

    avg_original, avg_fpc = average_tpr_at_max_m(LLRs_original, LLRs, all_in_indices)
    return (TPR_concat_original, TPR_concat_pp_original,
            TPR_concat, TPR_concat_pp, avg_original, avg_fpc)


In [ ]:
# evaluate_seed returns four budget curves followed by two maximum-M references.
panel_results = {}
for setting, (data_dir, dataset, n_records) in PANEL_SETTINGS.items():
    seed_results = [evaluate_seed(seed, data_dir, dataset, n_records) for seed in SEEDS]
    panel_results[setting] = tuple(
        np.stack([result[i] for result in seed_results]) for i in range(6)
    )
    print(f"{setting}: collected {len(SEEDS)} seed summaries")

In [ ]:
rc = bundles.iclr2024(usetex=False)
rc.update({
    # Set the line/bar color cycle (this is what affects ax.plot)
    "axes.prop_cycle": cycler(color=palette),
    # Optional readability tweaks
    "legend.frameon": False,
    "axes.grid": False,
})

titles = {
        "TabPFN": "Adult",
        "CIFAR10 + FiLM" : "CIFAR10 (FiLM)",
        "CIFAR10 + Head" : "CIFAR10 (Head)"
        }
with plt.rc_context(rc):
    fig, axes = plt.subplots(1, 3, sharex=True)
    for ax, setting in zip(axes, PANEL_SETTINGS):
        summary = panel_results[setting]
        for index, color, style, label in [
            (0, palette[0], ":", "Concatenated (Naive)"),
            (2, palette[0], "-", "Concatenated (Naive,FPC)"),
            (1, palette[1], ":", "Concatenated (PP)"),
            (3, palette[1], "-", "Concatenated (PP,FPC)"),
        ]:
            values = summary[index]
            ax.plot(Ms, np.median(values, axis=0), color=color, linestyle=style,
                    marker=".", label=label, alpha=0.7)
            ax.fill_between(Ms, values.min(axis=0), values.max(axis=0),
                            color=color, alpha=0.2)
        for index, style, condition in [(4, ":", "Pre-FPC"), (5, "-", "Post-FPC")]:
            values = summary[index]
            label = f"Avg. TPR/Sample (M={max(Ms)})" if condition == "Pre-FPC" else f"Avg. TPR/Sample (M={max(Ms)}, FPC)"
            ax.axhline(np.median(values), linestyle=style, color=palette[2],
                       label=label, alpha=0.7)
            ax.axhspan(values.min(), values.max(), color=palette[2],alpha=0.15)
        ax.set_title(titles[setting])
        ax.set_xscale("log", base=2)
        ax.set_yscale("log")
        ax.set_xticks(Ms[::2])
        # ax.xaxis.set_major_formatter(ScalarFormatter())
        ax.set_box_aspect(1)
    axes[0].set_ylabel(f"TPR @ FPR = {ALPHA:g}")
    axes[1].set_xlabel(r"$M$")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=3, frameon=False,
               bbox_to_anchor=(0.55,0.1))
    if SAVE_FIGURE:
        OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(OUTPUT_PATH, bbox_inches="tight")
    plt.show()